In [24]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# === Load Qwen ===
qwen_name = "Qwen/Qwen-7B"
qwen_tok = AutoTokenizer.from_pretrained(qwen_name, trust_remote_code=True)
qwen = AutoModelForCausalLM.from_pretrained(
    qwen_name, trust_remote_code=True, device_map="auto"
).eval()

# === Function to read translations ===
def load_translations(filepath, n_languages=None):
    with open(filepath, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f.readlines() if line.strip()]
    if n_languages is not None:
        lines = lines[:n_languages]
    print(f"Loaded {len(lines)} translations from {filepath}")
    for i, t in enumerate(lines, 1):
        print(f"  [{i}] {t}")
    return lines

# === Function to get hidden activations for one sentence ===
def get_hidden_for_sentence(sentence: str, layer_idx: int, last_token_only=False):
    q_inputs = qwen_tok(sentence, return_tensors="pt").to(device)
    hidden_cache = {}

    # detect layers
    if hasattr(qwen, "transformer") and hasattr(qwen.transformer, "h"):
        layers = qwen.transformer.h
    elif hasattr(qwen, "model") and hasattr(qwen.model, "decoder"):
        layers = qwen.model.decoder.layers
    else:
        raise ValueError("Cannot find transformer layers in Qwen model.")

    def hook_fn(module, inp, out):
        hidden_cache["h"] = out.detach()[0]

    handle = layers[layer_idx].register_forward_hook(hook_fn)
    with torch.no_grad():
        _ = qwen(**q_inputs)
    handle.remove()

    h = hidden_cache["h"]  # [seq_len, hidden_dim]
    return h[-1] if last_token_only else h.mean(dim=0)

# === Build centroid T ===
def build_centroid_T_from_file(filepath, layer_idx, n_languages=None, last_token_only=False):
    translations = load_translations(filepath, n_languages=n_languages)
    all_vecs = []
    for t in translations:
        vec = get_hidden_for_sentence(t, layer_idx, last_token_only=last_token_only)
        all_vecs.append(vec)
    T = torch.stack(all_vecs).mean(dim=0)
    T = T / (T.norm() + 1e-8)
    print(f"Built T with {len(all_vecs)} languages, shape {tuple(T.shape)}, norm {T.norm().item():.4f}")
    return T.to(device)

# === Steering generation ===
def generate_with_steering(input_en: str, T: torch.Tensor, layer_idx: int, max_new_tokens=20):
    q_inputs = qwen_tok(input_en, return_tensors="pt").to(device)

    def steering_hook(module, inp, out):
        h = out.clone()
        last = h[:, -1, :]
        proj = (last * T).sum(dim=-1, keepdim=True) * T
        h[:, -1, :] = last - proj
        return h

    if hasattr(qwen, "transformer") and hasattr(qwen.transformer, "h"):
        layers = qwen.transformer.h
    elif hasattr(qwen, "model") and hasattr(qwen.model, "decoder"):
        layers = qwen.model.decoder.layers

    handle = layers[layer_idx].register_forward_hook(steering_hook)

    with torch.no_grad():
        out_ids = qwen.generate(**q_inputs, max_new_tokens=max_new_tokens)
    handle.remove()
    return qwen_tok.decode(out_ids[0], skip_special_tokens=True)

# === Run test ===
if __name__ == "__main__":
    translations_path = "/home/acevedo/syn-sem/datasets/txt/steering/translations.txt"
    input_sentence = "Alessandro went for a hike on Sunday."
    total_layers = len(qwen.transformer.h) if hasattr(qwen, "transformer") else len(qwen.model.decoder.layers)
    central_layer = total_layers // 2
    print("Central layer:", central_layer)

    # Choose how many languages to use
    n_languages = 2  # try 1 for Spanish only, 2 for Spanish+Italian, etc.
    last_token_only = False  # can set True to use only final token activations
    T = build_centroid_T_from_file(translations_path, central_layer, n_languages=n_languages, last_token_only=last_token_only)

    # Baseline vs steered
    baseline = qwen.generate(**qwen_tok(input_sentence, return_tensors="pt").to(device), max_new_tokens=20)
    baseline_text = qwen_tok.decode(baseline[0], skip_special_tokens=True)
    steered = generate_with_steering(input_sentence, T, central_layer, max_new_tokens=20)

    print("\nBaseline:\n", baseline_text)
    print("\nSteered:\n", steered)


Loading checkpoint shards: 100%|██████████| 8/8 [02:49<00:00, 21.15s/it]


Central layer: 16
Loaded 2 translations from /home/acevedo/syn-sem/datasets/txt/steering/translations.txt
  [1] Alessandro fue de excursión el domingo.
  [2] Alessandro è andato a fare un'escursione domenica.


AttributeError: 'tuple' object has no attribute 'detach'